### **Section 1 - Import Libraries**

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import sys
from torch.utils.data import DataLoader
from torchvision import models

### **Section 2 - Device Configuration**

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple GPU (MPS)")
else:
    device = torch.device("cpu")
    print("Using CPU")

Using device: cpu
Using Apple GPU (MPS)


### **Section 3 - File Path and Pipeline Configuration**

In [3]:
sys.path.append(os.path.abspath(".."))

In [4]:
from src.data_loader import *
from src.transforms import *
from src.models import *
from src.train import *
from src.evaluate import *
print("Data Loader Imported Successfully")

Train Shape: (9580, 48, 48, 3)
Validation Shape: (2395, 48, 48, 3)
Test Shape: (6065, 48, 48, 3)
Unique Labels: [0 1 2]
Converted Train Labels: [0 1]
Converted Validation Labels: [0 1]
Converted Test Labels: [0 1]
Train Label Distribution:
[3200 6380]
Validation Label Distribution:
[ 875 1520]
Test Label Distribution:
[2245 3820]
Tomato Images: 20955
Maize Images: 400
Tomato Labels: [0 1]
Maize Labels: [0 1]
Tomato Train: 14668
Tomato Validation: 3143
Tomato Test: 3144
Maize Train: 280
Maize Validation: 60
Maize Test: 60
Using Apple GPU (MPS)
Feature Extractors Created
Data Loader Imported Successfully


- In ensemble learning the model combine final predictions.

- In Feaure fusion model will combine learned features, cross dataset feature sharing and domain adaptation which combines and 
 after that fusion model will train on that and gives the prediction accordingly.



### **Section 4 - Create Fusion Model**

In [5]:
fusion_model = FusionModel().to(device)

print("Fusion Model Created Successfully")

Fusion Model Created Successfully


### **Section 5 - Create Datasets and Dataloaders**

In [6]:
# Create Datasets

#Train Datasets
tomato_train_dataset = LeafDataset(
    train_tomato_data, train_tomato_labels, transform)

maize_train_dataset = LeafDataset(
    train_maize_data, train_maize_labels, transform)

maize2_train_dataset = LeafDataset(
    train_maize2_data, train_maize2_labels, transform)


#Validation Datasets
tomato_val_dataset = LeafDataset(
    val_tomato_data, val_tomato_labels, transform)

maize_val_dataset = LeafDataset(
    val_maize_data, val_maize_labels, transform)

maize2_val_dataset = LeafDataset(
    val_maize2_data, val_maize2_labels, transform)


#Test Datasets
tomato_test_dataset = LeafDataset(
    test_tomato_data, test_tomato_labels, transform)

maize_test_dataset = LeafDataset(
    test_maize_data, test_maize_labels, transform)

maize2_test_dataset = LeafDataset(
    test_maize2_data, test_maize2_labels, transform)


In [7]:
#Create DataLoaders

fusion_batch_size = 16

#Train Dataloaders
fusion_tomato_loader = DataLoader(
    tomato_train_dataset, batch_size=fusion_batch_size, shuffle=True, drop_last=True
)

fusion_maize_loader = DataLoader(
    maize_train_dataset, batch_size=fusion_batch_size, shuffle=True, drop_last=True
)

fusion_maize2_loader = DataLoader(
    maize2_train_dataset, batch_size=fusion_batch_size, shuffle=True, drop_last=True
)

#Validation Dataloaders
fusion_tomato_val_loader = DataLoader(
    tomato_val_dataset, batch_size=fusion_batch_size, shuffle=False, drop_last=True
)

fusion_maize_val_loader = DataLoader(
    maize_val_dataset, batch_size=fusion_batch_size, shuffle=False, drop_last=True
)

fusion_maize2_val_loader = DataLoader(
    maize2_val_dataset, batch_size=fusion_batch_size, shuffle=False, drop_last=True
)

#Test Dataloaders
fusion_tomato_test_loader = DataLoader(
    tomato_test_dataset, batch_size=fusion_batch_size, shuffle=False, drop_last=True
)

fusion_maize_test_loader = DataLoader(
    maize_test_dataset, batch_size=fusion_batch_size, shuffle=False, drop_last=True
)

fusion_maize2_test_loader = DataLoader(
    maize2_test_dataset, batch_size=fusion_batch_size, shuffle=False, drop_last=True
)

### **Section 6 - Fusion Optimiser**

In [8]:
fusion_criterion = nn.CrossEntropyLoss()

fusion_optimizer = torch.optim.Adam(
    fusion_model.parameters(),
    lr=0.0001
)

### **Section 7 - Train Feature Fusion Model**

In [9]:
fusion_train_losses, fusion_train_accuracies, fusion_val_accuracies = train_fusion_model(
    fusion_model,
    fusion_tomato_loader,
    fusion_maize_loader,
    fusion_maize2_loader,
    fusion_tomato_val_loader,
    fusion_maize_val_loader,
    fusion_maize2_val_loader,
    fusion_optimizer,
    fusion_criterion,
    device,
    epochs=10
)

Epoch [1/10]
Fusion Loss: 0.0112
Train Accuracy: 70.22%
Validation Accuracy: 87.50%
--------------------------------------------------
Epoch [2/10]
Fusion Loss: 0.0076
Train Accuracy: 78.68%
Validation Accuracy: 93.75%
--------------------------------------------------
Epoch [3/10]
Fusion Loss: 0.0038
Train Accuracy: 94.85%
Validation Accuracy: 100.00%
--------------------------------------------------
Epoch [4/10]
Fusion Loss: 0.0017
Train Accuracy: 97.43%
Validation Accuracy: 100.00%
--------------------------------------------------
Epoch [5/10]
Fusion Loss: 0.0010
Train Accuracy: 98.16%
Validation Accuracy: 100.00%
--------------------------------------------------
Epoch [6/10]
Fusion Loss: 0.0007
Train Accuracy: 98.53%
Validation Accuracy: 100.00%
--------------------------------------------------
Epoch [7/10]
Fusion Loss: 0.0010
Train Accuracy: 98.16%
Validation Accuracy: 100.00%
--------------------------------------------------
Epoch [8/10]
Fusion Loss: 0.0003
Train Accuracy: 9

### **Section 8 - Save and Load Models**

In [10]:
#Save the trained model

os.makedirs("saved_models", exist_ok=True)

# Save trained model
torch.save(
    fusion_model.state_dict(),
    "saved_models/fusion_model.pth"
)

print("Fusion model saved successfully")

Fusion model saved successfully


In [11]:
# Load the saved model

fusion_model.load_state_dict(
    torch.load(
        "saved_models/fusion_model.pth",
        map_location=device
    )
)

# Set models to evaluation mode
fusion_model.eval()

print("Fusion model loaded successfully")

Fusion model loaded successfully


### **Section 9 - Test Feature Fusion Model**

In [12]:
fusion_acc = evaluate_fusion_model(
    fusion_model,
    fusion_tomato_test_loader,
    fusion_maize_test_loader,
    fusion_maize2_test_loader,
    device
)

Feature Fusion Accuracy: 100.00%
